# Week 12 — The Capstone (STARTER)
### The Computer Vision Internship | VisionAI Lagos

Amara's Slack message (Monday 8:55am): 'Week 12. New dataset. You work alone. No questions until Friday.'

**Write your problem statement before loading any data.**

## Problem Statement

**What are we evaluating?** [ONE SENTENCE]

**What does success look like?** [CONCRETE F1 THRESHOLD]

**Prediction:** does the model generalise to a new Abuja sample with seed=999? [WHY OR WHY NOT]

**What could go wrong?**
1. [WRITE HERE]
2. [WRITE HERE]

In [ ]:
import sys,subprocess,os,warnings; warnings.filterwarnings("ignore")
def ensure(*p):
    for pkg in p:
        try: __import__(pkg.replace("-","_").split(".")[0])
        except ImportError: subprocess.check_call([sys.executable,"-m","pip","install",pkg,"-q"])
ensure("torch","torchvision","Pillow","numpy","pandas","matplotlib","seaborn","scikit-learn","tqdm")
import torch,torch.nn as nn,numpy as np,pandas as pd,matplotlib.pyplot as plt,seaborn as sns
from torchvision import transforms,models; from torch.utils.data import Dataset,DataLoader
from PIL import Image; from pathlib import Path; from sklearn.metrics import f1_score,classification_report,confusion_matrix
torch.manual_seed(42); np.random.seed(42)
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
MEAN=[0.485,0.456,0.406]; STD=[0.229,0.224,0.225]
CLASSES=["commercial","green_space","industrial","residential"]
CLS2IDX={c:i for i,c in enumerate(CLASSES)}; IDX2CLS={v:k for k,v in CLS2IDX.items()}
os.makedirs("outputs",exist_ok=True)

class SatelliteDataset(Dataset):
    def __init__(self,df,img_dir,transform=None):
        self.df=df.reset_index(drop=True); self.img_dir=Path(img_dir); self.transform=transform
    def __len__(self): return len(self.df)
    def __getitem__(self,idx):
        row=self.df.iloc[idx]; img=Image.open(self.img_dir/row["filename"]).convert("RGB")
        return (self.transform(img) if self.transform else img),CLS2IDX[row["land_use"]],row["city"]

def build_efficientnet_b0(nc=4):
    m=models.efficientnet_b0(weights=None)
    m.classifier=nn.Sequential(nn.Dropout(0.3),nn.Linear(1280,nc)); return m

model=build_efficientnet_b0().to(DEVICE)
model.load_state_dict(torch.load("models/efficientnet_best.pth",map_location=DEVICE))
model.eval(); ev=transforms.Compose([transforms.ToTensor(),transforms.Normalize(MEAN,STD)])
print(f"Model loaded ✅ | Device: {DEVICE}")

## Step 1 — Generate Capstone Corpus

> 🧠 **What Will This Output?**
> The capstone corpus uses seed=999; training used seed=42. Does the seed difference matter for model performance? Should two batches of Abuja patches from the same generator look meaningfully different to the model?

In [ ]:
import os, subprocess, sys

# Generate synthetic dataset if not present
DATA_DIR = 'datasets'
if not os.path.exists(DATA_DIR) or not os.listdir(DATA_DIR):
    print('Generating dataset...')
    if os.path.exists('generate_images.py'):
        subprocess.run([sys.executable, 'generate_images.py'], check=False)
    else:
        os.makedirs(DATA_DIR, exist_ok=True)
        print('⚠️  generate_images.py not found. Run from the repo root.')
        print('   git clone https://github.com/internshipinabook/cv-internship-in-a-book-in-a-book.git')
else:
    print('Dataset ready ✅')

# Verify dataset
from pathlib import Path
img_count = len(list(Path(DATA_DIR).rglob('*.png'))) + len(list(Path(DATA_DIR).rglob('*.jpg')))
print(f'Images found: {img_count:,}')


## Step 2 — Evaluate

In [ ]:
loader=DataLoader(SatelliteDataset(cap_df,"datasets/capstone_abuja/images",ev),32,False,num_workers=0)
preds,labels=[],[]
with torch.no_grad():
    for imgs,lbls,_ in loader:
        preds.extend(model(imgs.to(DEVICE)).argmax(1).cpu().numpy()); labels.extend(lbls.numpy())
cap_f1=f1_score(labels,preds,average="weighted",zero_division=0)
print(f"Capstone Abuja F1: {cap_f1:.3f}")
print()
print(classification_report(labels,preds,target_names=CLASSES,zero_division=0))

cm=confusion_matrix(labels,preds)
fig,ax=plt.subplots(figsize=(6,5))
sns.heatmap(pd.DataFrame(cm,index=CLASSES,columns=CLASSES),annot=True,fmt="d",cmap="Blues",ax=ax)
ax.set_title(f"Capstone Confusion Matrix (F1={cap_f1:.3f})",fontweight="bold",loc="left")
plt.tight_layout(); plt.savefig("outputs/capstone_cm.png",dpi=150,bbox_inches="tight",facecolor="white"); plt.show()

## Step 3 — GradCAM on Correct and Incorrect Predictions

In [ ]:
class GradCAM:
    def __init__(self,model,target_layer):
        self.model=model; self.activations=None; self.gradients=None
        target_layer.register_forward_hook(lambda m,i,o: setattr(self,"activations",o.detach()))
        target_layer.register_full_backward_hook(lambda m,gi,go: setattr(self,"gradients",go[0].detach()))
    def generate(self,img_t,class_idx=None):
        self.model.eval(); out=self.model(img_t)
        if class_idx is None: class_idx=out.argmax(dim=1).item()
        self.model.zero_grad()
        oh=torch.zeros_like(out); oh[0,class_idx]=1.0; out.backward(gradient=oh,retain_graph=True)
        w=self.gradients.mean(dim=[2,3],keepdim=True)
        cam=torch.relu((w*self.activations).sum(dim=1)).squeeze().cpu().numpy()
        if cam.ndim==0: cam=np.array([[float(cam)]])
        from PIL import Image as PI
        cam_r=np.array(PI.fromarray(cam).resize((img_t.shape[-1],img_t.shape[-2])))
        return (cam_r/cam_r.max() if cam_r.max()>0 else cam_r),class_idx

In [ ]:
gradcam=GradCAM(model,model.features[-1])
cap_df2=cap_df.copy(); cap_df2["pred"]=preds; cap_df2["true"]=labels
cap_df2["correct"]=cap_df2["pred"]==cap_df2["true"]

correct_samp=cap_df2[cap_df2["correct"]].sample(min(2,cap_df2["correct"].sum()),random_state=42)
wrong_samp  =cap_df2[~cap_df2["correct"]].head(2)
plot_rows=pd.concat([correct_samp,wrong_samp])

if len(plot_rows)>0:
    fig,axes=plt.subplots(len(plot_rows),3,figsize=(12,4*len(plot_rows)))
    if len(plot_rows)==1: axes=[axes]
    for ri,(_,row) in enumerate(plot_rows.iterrows()):
        img_pil=Image.open(f"datasets/capstone_abuja/images/{row['filename']}").convert("RGB")
        img_t=ev(img_pil).unsqueeze(0).to(DEVICE)
        cam,_=gradcam.generate(img_t,class_idx=int(row["true"]))
        img_arr=np.array(img_pil)
        axes[ri][0].imshow(img_arr)
        axes[ri][0].set_title(f"True:{IDX2CLS[row['true']]}",color="green" if row["correct"] else "red",fontsize=9)
        axes[ri][0].axis("off")
        axes[ri][1].imshow(cam,cmap="jet"); axes[ri][1].set_title("GradCAM",fontsize=9); axes[ri][1].axis("off")
        axes[ri][2].imshow(img_arr); axes[ri][2].imshow(cam,cmap="jet",alpha=0.45)
        axes[ri][2].set_title(f"Pred:{IDX2CLS[row['pred']]}",fontsize=9); axes[ri][2].axis("off")
    plt.suptitle("Capstone GradCAM — Correct and Incorrect Predictions",fontweight="bold")
    plt.tight_layout(); plt.savefig("outputs/capstone_gradcam.png",dpi=150,bbox_inches="tight",facecolor="white"); plt.show()
else:
    print("No errors found — model classified all capstone patches correctly.")

## Step 4 — Reflection

In [ ]:
LETTER="""
Portfolio Letter — Week 12 Capstone

Project: VisionAI Lagos Land-Use Classifier
Corpus:  Abuja-only synthetic patches (seed=999)
Task:    Full evaluation + GradCAM analysis

Capstone F1: [FILL IN]

What I found:
[WRITE 2-3 sentences about the result]

What surprised me:
[WRITE 1-2 sentences]

What I would do differently:
[ONE specific technical decision]

What I carry into the next project:
[ONE habit, not a methodology]

The GradCAM visualisations are the most honest part of this notebook.
They show where the model pays attention — and whether that attention is justified.

--- [Your Name], [Date]
"""
with open("outputs/capstone_portfolio_letter.md","w") as f: f.write(LETTER)
print("Portfolio letter saved ✅")
print("Personalise every line — then add capstone.ipynb to your portfolio.")

## Certificate

```
+--------------------------------------------------+
|          CERTIFICATE OF COMPLETION              |
|   THE COMPUTER VISION INTERNSHIP — BOOK 3 OF 7 |
|                                                  |
|  _____________________________________           |
|  has completed twelve weeks of CV engineering   |
|  at VisionAI Lagos.                             |
|                                                  |
|  Date: ______________________                    |
+--------------------------------------------------+
```

---
## Book 3 Complete

*The Computer Vision Internship · Book 3 of 7 · Internship in a Book Series*

## ✅ By completing Week 12, you can now:

- Assemble a complete CV portfolio: notebooks, model card, fairness brief, GradCAM report
- Deploy the best model as an API endpoint with confidence threshold
- Write a retrospective identifying what you would do differently in Weeks 2, 5, and 9
- Push a recruiter-ready GitHub repo with v1.0 release tag

📤 **GitHub:** Final push: all notebooks, model card, fairness brief, API, README. Tag v1.0. Commit: "Week 12: Capstone complete"


---

## 📚 Get the Full Book

This notebook is part of **The Computer Vision Internship** — Book 3 of the InternshipInABook™ Series.

👉 **[Get the book on Selar → [SELAR_LINK_PLACEHOLDER]]**

---
*InternshipInABook™ · © Sakinat Folorunso 2026 · [github.com/internshipinabook](https://github.com/internshipinabook/cv-internship-in-a-book)*
